In [ ]:
import chrombpnet  # selects the Keras JAX backend; import it before keras
import chrombpnet.training.utils.one_hot as one_hot
from chrombpnet.training.utils.model_io import load_model
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ---- Step 1: Load model ----

MODEL_PATH = "results/chrombpnet/ATAC_PE/GM12878/nautilus_runs/GM12878_03.01.2022_bias_128_4_1234_0.4_fold_0/chrombpnet_model/chrombpnet_wo_bias.h5"  # Change this path to your actual model
model = load_model(MODEL_PATH)  # chrombpnet 1.x (TF-Keras) and 2.x .h5 files

In [ ]:
# ---- Step 2: Input your custom 2114bp sequence ----

# Make sure it only contains A, C, G, T
def load_sequences_from_fasta(fasta_path, seq_len):
    sequences = []
    with open(fasta_path) as f:
        current_seq = []
        for line in f:
            if line.startswith(">"):
                if current_seq:
                    sequences.append("".join(current_seq))
                    current_seq = []
            else:
                current_seq.append(line.strip())

        if current_seq:
            sequences.append("".join(current_seq))
    assert all(len(s) == seq_len for s in sequences), "Input sequences must be exactly {} bp long.".format(seq_len)
    return sequences

sequences = load_sequences_from_fasta("example.fa", model.input_shape[1])

# ---- Step 3: One-hot encode the sequence ----

one_hot_seqs = one_hot.dna_to_one_hot(sequences)  # shape: (N, 2114, 4)
print("One-hot shape:", one_hot_seqs.shape)

# ---- Step 4: Get prediction ----

pred_logits_wo_bias, pred_logcts_wo_bias = model.predict(one_hot_seqs.astype("float32"), verbose=0)

def softmax(x, temp=1):
    norm_x = x - np.mean(x,axis=1, keepdims=True)
    return np.exp(temp*norm_x)/np.sum(np.exp(temp*norm_x), axis=1, keepdims=True)

predictions = softmax(pred_logits_wo_bias) * (np.expand_dims(np.exp(pred_logcts_wo_bias)[:,0],axis=1)) # final predcitions you can use

print("Prediction logits shape:", pred_logits_wo_bias.shape) # (N, 1000)
print("Prediction logcnt shape:", pred_logcts_wo_bias.shape) # (N, 1)
print("Prediction shape:", predictions.shape) # (N, 1000)


In [ ]:
index=0 # plot predictions for example 0
plt.plot(predictions[index])